In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser

# 1. Load & Split (run once)
loader = PyPDFLoader("./docs/Langchain_syllabus.pdf")  # replace with your file
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=80)
chunks = splitter.split_documents(docs)

# 2. Create Vector Store (run once)
embeddings = HuggingFaceEmbeddings(model_name="./embedding_model")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_rag_db"
)

# 3. Create Retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# 4. Build the RAG Pipeline
llm = ChatGoogleGenerativeAI(
    model="gemini-flash-latest",
    temperature=0.1,
    streaming=True
)

template = """You are a helpful assistant. Answer the question based **only** on the following context.
If you don't know the answer, say "I don't have enough information."

Context:
{context}

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n---\n\n".join(
        f"Source: {doc.metadata.get('source', 'unknown')}\n{doc.page_content}"
        for doc in docs
    )

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# Run it
response = rag_chain.invoke("What does the document say about text splitting?")
print(response)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Based on the provided document, text splitting includes the following:

* **Topics to Learn:** 
  * `RecursiveCharacterTextSplitter`
  * Chunk size strategies
  * Overlap tuning
  * Semantic chunking
* **Critical Discussion:** 
  * Chunking quality directly impacts RAG quality.
